## Классификация новостей [AG's News Topic Classification Dataset](https://github.com/mhjabreel/CharCnn_Keras/tree/master/data/ag_news_csv)

Используется три архитектуры нейронных сетей:
- Одномерная сверточная нейросеть
- Рекуррентная нейросеть LSTM
- Рекуррентная нейросеть GRU

Для ускорения обучения сети рекомендуется подключить GPU (Runtime -> Change Runtime Type -> Hardware Accelerator -> GPU).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras import utils
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import utils
from tensorflow.keras import regularizers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# Максимальное количество уникальных слов слов
num_words = 10000
# Максимальная длина новости
max_news_len = 50
# Количество классов новостей
nb_classes = 4

## Загрузка набора данных

Загружаем данные для обучения

In [ ]:
!wget https://github.com/mhjabreel/CharCnn_Keras/raw/master/data/ag_news_csv/train.csv -O train.csv

Загружаем данные для тестирования

In [ ]:
!wget https://github.com/mhjabreel/CharCnn_Keras/raw/master/data/ag_news_csv/test.csv -O test.csv

Загружаем имена классов

In [ ]:
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/classes.txt -O classes.txt

## Просматриваем данные

In [ ]:
!ls

In [ ]:
!cat classes.txt

In [ ]:
!head train.csv

In [ ]:
!head test.csv

In [ ]:
!wc -l train.csv
!wc -l test.csv

## Загружаем данные в память

Читаем данные из файла

In [ ]:
train = pd.read_csv('train.csv',
                    header=None,
                    names=['class', 'title', 'text'])

In [ ]:
train

Выделяем данные для обучения

In [ ]:
import os
import nltk
# Скачиваем punkt в локальную папку
nltk.download('punkt', download_dir='/content/nltk_data')

# Добавляем эту папку в пути поиска NLTK
nltk.data.path.append('/content/nltk_data')

In [ ]:

nltk.download('all')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from string import punctuation

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """
    1) Токенизируем (разбиваем на слова)
    2) Приводим к нижнему регистру
    3) Убираем пунктуацию и стоп-слова
    4) Лемматизируем
    5) Склеиваем обратно в строку
    """
    tokens = nltk.word_tokenize(text.lower())
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and token not in punctuation:
            lemma = lemmatizer.lemmatize(token)
            cleaned_tokens.append(lemma)
    return " ".join(cleaned_tokens)

In [ ]:
# news = train['text']
news = train['title'] + ' ' + train['text']
news = news.apply(preprocess_text)

In [ ]:
news[:5]

Выделяем правильные ответы

In [ ]:
# Преобразование целочисленного вектора класса в двоичную матрицу
# Аргументы: вектор классов в данных и общее количество классов
y_train = utils.to_categorical(train['class'] - 1, nb_classes)

In [ ]:
y_train

## Токенизация текста

In [ ]:
news[:5]

Создаем токенизатор Keras

<img src="https://habrastorage.org/r/w1560/getpro/habr/upload_files/bc9/feb/314/bc9feb3143af4759aceff629305cf8ae.png">

Принцип работы модуля TextVectorization

In [ ]:
vectorize_layer = tf.keras.layers.TextVectorization(max_tokens=num_words,
    output_mode='int',
    output_sequence_length=max_news_len)

In [ ]:
vectorize_layer.adapt(news)
vectorize_layer.get_vocabulary()

In [ ]:
tokenizer = Tokenizer(num_words=num_words)

Обучаем токенизатор на новостях

In [ ]:
tokenizer.fit_on_texts(news)

Просматриваем словарь токенизатора

In [ ]:
tokenizer.word_index

Преобразуем новости в числовое представление

In [ ]:
sequences = tokenizer.texts_to_sequences(news)

Просматриваем новости в числовом представлении

In [ ]:
index = 1
print(news[index])
print(sequences[index])

In [ ]:
tokenizer.word_index['reuters']

Ограничиваем длину текста

In [ ]:
x_train = pad_sequences(sequences, maxlen=max_news_len)

In [ ]:
x_train[:5]

## Сверточная нейронная сеть

In [ ]:
model_cnn = keras.Sequential()
model_cnn.add(layers.Embedding(num_words, 32, input_length=max_news_len))
model_cnn.add(layers.Conv1D(
    filters=250,
    kernel_size=5,
    padding='valid',
    activation='relu',
    kernel_regularizer=regularizers.l2(1e-4)  # добавляем L2-регуляризацию (пример)
))
model_cnn.add(layers.GlobalMaxPooling1D())

# Добавляем Dropout перед полносвязными слоями
model_cnn.add(layers.Dropout(0.5))  # "выключаем" 50% нейронов
model_cnn.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)))
model_cnn.add(layers.Dropout(0.5))
model_cnn.add(layers.Dense(4, activation='softmax'))

In [ ]:
model_cnn.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model_cnn.summary()

Создаем callback для сохранения нейронной сети на каждой эпохе, если качество работы на проверочном наборе данных улучшилось. Сеть сохраняется в файл `best_model.h5`

In [ ]:
# model_cnn_save_path = 'best_model_cnn.h5'
checkpoint_filepath_cnn = '/tmp/ckpt/checkpoint.model_cnn.keras'
checkpoint_callback_cnn = ModelCheckpoint(checkpoint_filepath_cnn,
                                      monitor='val_accuracy',
                                      save_best_only=True,
                                      verbose=1)

In [ ]:
history_cnn = model_cnn.fit(x_train,
                            y_train,
                            epochs=5,
                            batch_size=128,
                            validation_split=0.1,
                            callbacks=[checkpoint_callback_cnn])

In [ ]:
plt.plot(history_cnn.history['accuracy'],
         label='Доля верных ответов на обучающем наборе')
plt.plot(history_cnn.history['val_accuracy'],
         label='Доля верных ответов на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Доля верных ответов')
plt.legend()
plt.show()

## Cеть LSTM

In [ ]:
model_lstm = keras.Sequential()
model_lstm.add(layers.Embedding(num_words, 32, input_length=max_news_len))
model_lstm.add(layers.LSTM(
    16,
    dropout=0.5,   # Dropout для входных связей
    recurrent_dropout=0.5  # Dropout для рекуррентных связей
))
model_lstm.add(layers.Dense(4, activation='softmax'))

In [ ]:
model_lstm.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model_lstm.summary()

Создаем callback для сохранения нейронной сети на каждой эпохе, если качество работы на проверочном наборе данных улучшилось. Сеть сохраняется в файл `best_model.h5`

In [ ]:
# model_lstm_save_path = 'best_model_lstm.h5'
checkpoint_filepath_lstm = '/tmp/ckpt/checkpoint.model_lstm.keras'
checkpoint_callback_lstm = ModelCheckpoint(checkpoint_filepath_lstm,
                                      monitor='val_accuracy',
                                      save_best_only=True,
                                      verbose=1)

In [ ]:
history_lstm = model_lstm.fit(x_train,
                              y_train,
                              epochs=5,
                              batch_size=128,
                              validation_split=0.1,
                              callbacks=[checkpoint_callback_lstm])

In [ ]:
plt.plot(history_lstm.history['accuracy'],
         label='Доля верных ответов на обучающем наборе')
plt.plot(history_lstm.history['val_accuracy'],
         label='Доля верных ответов на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Доля верных ответов')
plt.legend()
plt.show()

## Cеть GRU

In [ ]:
model_gru = keras.Sequential()
model_gru.add(layers.Embedding(num_words, 32, input_length=max_news_len))
model_gru.add(layers.GRU(
    16,
    dropout=0.5,   # Dropout для входных связей
    recurrent_dropout=0.5  # Dropout для рекуррентных связей
))
model_gru.add(layers.Dense(4, activation='softmax'))

In [ ]:
model_gru.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model_gru.summary()

Создаем callback для сохранения нейронной сети на каждой эпохе, если качество работы на проверочном наборе данных улучшилось. Сеть сохраняется в файл `best_model_gru.h5`

In [ ]:
# model_gru_save_path = 'best_model_gru.h5'
checkpoint_filepath_gru = '/tmp/ckpt/checkpoint.model_gru.keras'
checkpoint_callback_gru = ModelCheckpoint(checkpoint_filepath_gru,
                                      monitor='val_accuracy',
                                      save_best_only=True,
                                      verbose=1)

In [ ]:
history_gru = model_gru.fit(x_train,
                              y_train,
                              epochs=5,
                              batch_size=128,
                              validation_split=0.1,
                              callbacks=[checkpoint_callback_gru])

In [ ]:
plt.plot(history_gru.history['accuracy'],
         label='Доля верных ответов на обучающем наборе')
plt.plot(history_gru.history['val_accuracy'],
         label='Доля верных ответов на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Доля верных ответов')
plt.legend()
plt.show()

## Загружаем набор данных для тестирования

In [ ]:
test = pd.read_csv('test.csv',
                    header=None,
                    names=['class', 'title', 'text'])

In [ ]:
test

Преобразуем новости в числовое представление

Обратите внимание, что нужно использовать токенизатор, обученный на наборе данных train.

In [ ]:
news =  test['title'] + ' ' + test['text']
news = news.apply(preprocess_text)
test_sequences = tokenizer.texts_to_sequences(news)

In [ ]:
x_test = pad_sequences(test_sequences, maxlen=max_news_len)

In [ ]:
x_test[:5]

Правильные ответы

In [ ]:
y_test = utils.to_categorical(test['class'] - 1, nb_classes)

In [ ]:
y_test

## Оцениваем качество работы сети на тестовом наборе данных

### Одномерная сверточная сеть

In [ ]:
model_cnn.load_weights(checkpoint_filepath_cnn)

In [ ]:
model_cnn.evaluate(x_test, y_test, verbose=1)

### Сеть LSTM

In [ ]:
model_lstm.load_weights(checkpoint_filepath_lstm)

In [ ]:
model_lstm.evaluate(x_test, y_test, verbose=1)

### Сеть GRU

In [ ]:
model_gru.load_weights(checkpoint_filepath_gru)

In [ ]:
model_gru.evaluate(x_test, y_test, verbose=1)

## Задание

1. Попробовать улучшить предсказание моделей за счет использования не только текста, но и заголовка новости.
При этом рекомендуется увеличить максимальню длину текста. <br/>
2. Проверить, как на качеаство предсказания влияет предобработка текста (удаление стоп-слов, пунктуации, лемматизация и др.)
3. Применить методы регуляризации для устранения переобучения моделей.


## Стоп-слова и пунктуация

**Стоп-слова** - это слова, которые часто встречаются практически в любом тексте и не несут полезной информации о конретном документе. Для модели это просто шум. А шум нужно убирать. По аналогичной причине убирают и пунктуацию.

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

In [ ]:
# импортируем стоп-слова из библиотеки nltk
from nltk.corpus import stopwords

# посмотрим на стоп-слова для английского языка
print(stopwords.words('english'))

*Знаки* пунктуации лучше импортировать из модуля **String**. В нем хранятся различные наборы констант для работы со строками (пунктуация, алфавит и др.).

In [ ]:
from string import punctuation
punctuation



Объединим стоп-слова и знаки пунктуации вместе и запишем в переменную noise:


In [ ]:
noise = stopwords.words('english') + list(punctuation)

## Лемматизация

**Лемматизация** – это сведение разных форм одного слова к начальной форме – **лемме**. Почему это хорошо?
* Во-первых, естественно рассматривать как отдельный признак каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
news = list(lemmatizer.lemmatize(word) for word in news)

## Полезные ссылки

1. [Как понять LSTM сети](https://alexsosn.github.io/ml/2015/11/17/LSTM.html)
2. [Рекуррентные нейронные сети в Keras](https://keras.io/layers/recurrent/)
3. [Регуляризуем правильно!](https://telegra.ph/Regulyarizuem-pravilno-09-20)
4. [12 основных методов Dropout](https://towardsdatascience.com/12-main-dropout-methods-mathematical-and-visual-explanation-58cdc2112293?source=topic_page---------6------------------1)